In [1]:
from elasticsearch import Elasticsearch

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env into environment

es = Elasticsearch(
    os.environ["ES_URL"],
    basic_auth=(
        os.environ["ES_USERNAME"],
        os.environ["ES_PASSWORD"]
    ),
    ca_certs=os.environ["ES_CA_CERT"]
)


## Prepare data

In [3]:
import pandas as pd
df=pd.read_csv("/home/sonu/Desktop/semantic_search_engine/myntra_products_catalog.csv").loc[:499]

In [4]:
df.head()

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White


In [5]:
df.isnull().sum()

ProductID        0
ProductName      0
ProductBrand     0
Gender           0
Price (INR)      0
NumImages        0
Description      0
PrimaryColor    32
dtype: int64

In [6]:
df.fillna("None",inplace=True)

## Convert to vectors

In [7]:
from sentence_transformers import SentenceTransformer

# Load https://huggingface.co/sentence-transformers/all-mpnet-base-v2
model = SentenceTransformer("all-mpnet-base-v2")

/home/sonu/Desktop/semantic_search_engine/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 561.24it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
descriptions = df["Description"].tolist()

embeddings = model.encode(
    descriptions,
    batch_size=32,     
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


Batches: 100%|██████████| 16/16 [00:40<00:00,  2.54s/it]


In [9]:
for i, row in df.iterrows():
    es.index(
        index="all_products",
        document={
            "ProductID": int(row["ProductID"]),
            "ProductName": row["ProductName"],
            "ProductBrand": row["ProductBrand"],
            "Gender": row["Gender"],
            "Price (INR)": int(row["Price (INR)"]),
            "NumImages": int(row["NumImages"]),
            "Description": row["Description"],
            "PrimaryColor": row["PrimaryColor"],
            "DescriptionVector": embeddings[i].tolist()
        }
    )


In [10]:
del embeddings
